
 ## SparqlCE training script — DBpedia.

Reproduces the paper's main-branch recipe:
    loss      = InfoNCE(in-batch) + 0.3 * MSE(cos(a,p), jaccard(a,p))
    optimizer = AdamW, differential LRs (CodeBERT 1e-5, GNN/proj 1e-4)
                (or single LR 1e-5 — flip USE_DIFFERENTIAL_LR below)
    selection = best val MRR  (NOT mean cosine — that's biased toward collapse)
    splits    = paper's fixed split if available, else seeded random split

Run:
    python train_dbpedia.py

Before running, VERIFY the three paths in the CONFIG block.
"""



In [5]:

import os
import sys
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from tqdm import tqdm


# ============================================================================
# 1. CONFIG — verify these three paths match your repo before running
# ============================================================================
DATA_PATH   = "../../datasets/dbpedia-triplets-dataset.json"
SPLITS_PATH = "../../datasets/splits_dbpedia.json"  # paper's fixed split
CKPT_DIR    = "../../checkpoints"

# Hyperparameters that match the paper / main branch
SEED         = 42
EPOCHS       = 20            # paper: 20
BATCH_SIZE   = 32            # larger batch = more in-batch negatives
TEMPERATURE  = 0.05          # paper: 0.05  (NOT 0.07)
LAMBDA_REG   = 0.3           # Jaccard calibration weight
WEIGHT_DECAY = 1e-2
GRAD_CLIP    = 1.0

In [6]:
from pathlib import Path

# Differential LRs: fine-tune CodeBERT gently, train GNN/proj at a normal rate.
# Set USE_DIFFERENTIAL_LR = False to use a single 1e-5 for everything (safest,
# matches the paper exactly but slower for the random-init GNN to converge).
USE_DIFFERENTIAL_LR = True
LR_TEXT_ENCODER     = 1e-5
LR_REST             = 1e-4
LR_SINGLE           = 1e-5


# ============================================================================
# 2. Setup
# ============================================================================
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Path.cwd() gets the folder where your notebook is currently running
# .parent.parent moves up two directories
PROJECT_ROOT = str(Path.cwd().parent.parent)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from sparqlce.config import Config  # noqa: E402  (kept in case you reference Config elsewhere)
from sparqlce.graph_builder import SPARQLGraphBuilder  # noqa: E402
from sparqlce.encoder import HybridSPARQLEncoder  # noqa: E402
from sparqlce.dataset import (  # noqa: E402
    SemanticSPARQLDataset,
    semantic_collate_fn_inbatch,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [7]:

# ============================================================================
# 3. Data: dataset + fixed split (paper) with safe random fallback
# ============================================================================
graph_builder = SPARQLGraphBuilder()
dataset = SemanticSPARQLDataset(
    DATA_PATH,
    graph_builder=graph_builder,
    in_batch_negatives=True,    # yields (anchor, positive, jaccard) tuples
)

if os.path.exists(SPLITS_PATH):
    with open(SPLITS_PATH, "r") as f:
        splits = json.load(f)
    # Tolerant key handling so this works across split-file variants
    train_idx = splits.get("train_idx", splits.get("train"))
    val_idx   = splits.get("val_idx",   splits.get("valid_idx", splits.get("val")))
    test_idx  = splits.get("test_idx",  splits.get("test"))
    assert train_idx and val_idx and test_idx, f"Could not parse splits from {SPLITS_PATH}"

    train_dataset = Subset(dataset, train_idx)
    val_dataset   = Subset(dataset, val_idx)
    test_dataset  = Subset(dataset, test_idx)
    print(f"✓ Using FIXED split from {SPLITS_PATH}")
else:
    print(f"⚠  Fixed split file not found at {SPLITS_PATH}")
    print(f"⚠  Falling back to seeded random_split — numbers will NOT match paper Table 2.")
    print(f"⚠  Locate the paper's split file before reporting results.")
    total = len(dataset)
    n_train = int(0.80 * total)
    n_val   = int(0.10 * total)
    n_test  = total - n_train - n_val
    gen = torch.Generator().manual_seed(SEED)
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [n_train, n_val, n_test], generator=gen
    )

print(f"  train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=semantic_collate_fn_inbatch)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=semantic_collate_fn_inbatch)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=semantic_collate_fn_inbatch)


Loading dataset from ../../datasets/dbpedia-triplets-dataset.json...
Loaded 4713 anchor groups.
✓ Using FIXED split from ../../datasets/splits_dbpedia.json
  train=3770  val=471  test=472


In [8]:


# ============================================================================
# 4. Model + optimizer
# ============================================================================
model = HybridSPARQLEncoder().to(device)

if USE_DIFFERENTIAL_LR:
    # Group by parameter name: anything that looks like the text encoder gets a
    # gentle LR; everything else (R-GCN/GAT layers, projections) gets a larger one.
    text_kw = ("text", "txt", "codebert", "bert")
    text_params = [p for n, p in model.named_parameters()
                   if any(k in n.lower() for k in text_kw) and p.requires_grad]
    rest_params = [p for n, p in model.named_parameters()
                   if not any(k in n.lower() for k in text_kw) and p.requires_grad]
    n_text = sum(p.numel() for p in text_params)
    n_rest = sum(p.numel() for p in rest_params)
    print(f"✓ Differential LRs: text={n_text:,} params @ {LR_TEXT_ENCODER}, "
          f"other={n_rest:,} params @ {LR_REST}")
    if n_text == 0:
        print("⚠  No params matched text-encoder keywords; using single LR fallback.")
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR_SINGLE,
                                      weight_decay=WEIGHT_DECAY)
    else:
        optimizer = torch.optim.AdamW([
            {"params": text_params, "lr": LR_TEXT_ENCODER},
            {"params": rest_params, "lr": LR_REST},
        ], weight_decay=WEIGHT_DECAY)
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_SINGLE,
                                  weight_decay=WEIGHT_DECAY)


✓ Differential LRs: text=124,645,632 params @ 1e-05, other=2,494,592 params @ 0.0001


In [9]:


# ============================================================================
# 5. Losses
# ============================================================================
def infonce_inbatch(a_emb, p_emb, temperature):
    """In-batch InfoNCE. Matches the paper / main branch."""
    a = F.normalize(a_emb, dim=1)
    p = F.normalize(p_emb, dim=1)
    logits = a @ p.T / temperature
    labels = torch.arange(a.size(0), device=a.device)
    return F.cross_entropy(logits, labels)


In [10]:
# ============================================================================
# 6. Validation: rank each anchor's true positive against ALL positives in
# the val split. Selects the embedding that retrieves correctly, not the one
# that collapses everything to a single point.
# ============================================================================
@torch.no_grad()
def evaluate_ranking(model, loader, device):
    model.eval()
    anchors, positives = [], []
    for batch in loader:
        a_emb = F.normalize(model(batch[0].to(device)), dim=1)
        p_emb = F.normalize(model(batch[1].to(device)), dim=1)
        anchors.append(a_emb.cpu())
        positives.append(p_emb.cpu())
    A = torch.cat(anchors)      # [N, d]
    P = torch.cat(positives)    # [N, d]
    sims  = A @ P.T             # [N, N]; diagonal is the true positive
    diag  = sims.diag().unsqueeze(1)
    ranks = (sims >= diag).sum(dim=1).float()   # 1-indexed rank of each true positive
    return {
        "mrr":    (1.0 / ranks).mean().item(),
        "hits@1": (ranks == 1).float().mean().item(),
        "hits@5": (ranks <= 5).float().mean().item(),
    }



In [11]:



# ============================================================================
# 7. Training loop
# ============================================================================
os.makedirs(CKPT_DIR, exist_ok=True)
best_mrr = -1.0

print(f"\n✓ Training on {device} for {EPOCHS} epochs")
print(f"  loss = InfoNCE(τ={TEMPERATURE}) + {LAMBDA_REG} * MSE(cos, jaccard)")

for epoch in range(1, EPOCHS + 1):
    model.train()
    sum_loss, sum_ce, sum_reg = 0.0, 0.0, 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS}")

    for batch in pbar:
        a_graphs = batch[0].to(device)
        p_graphs = batch[1].to(device)
        pos_js   = batch[2].to(device)

        optimizer.zero_grad()
        a_emb = model(a_graphs)
        p_emb = model(p_graphs)

        # InfoNCE on the batch
        loss_ce = infonce_inbatch(a_emb, p_emb, TEMPERATURE)

        # Jaccard calibration MSE on positives.
        # Compute on cosine -> embeddings must be L2-normalised first.
        a_n = F.normalize(a_emb, dim=1)
        p_n = F.normalize(p_emb, dim=1)
        cos = (a_n * p_n).sum(dim=1)
        loss_reg = (cos - pos_js).pow(2).mean()

        loss = loss_ce + LAMBDA_REG * loss_reg
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        sum_loss += loss.item()
        sum_ce   += loss_ce.item()
        sum_reg  += loss_reg.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         ce=f"{loss_ce.item():.4f}",
                         reg=f"{loss_reg.item():.4f}")

    nb = max(1, len(train_loader))
    val = evaluate_ranking(model, val_loader, device)
    print(f"Epoch {epoch:02d} | train_loss={sum_loss/nb:.4f} "
          f"(ce={sum_ce/nb:.4f} reg={sum_reg/nb:.4f}) "
          f"| val MRR={val['mrr']:.4f}  Hits@1={val['hits@1']:.4f}  Hits@5={val['hits@5']:.4f}")

    if val["mrr"] > best_mrr:
        best_mrr = val["mrr"]
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_metrics": val,
        }, os.path.join(CKPT_DIR, "best_model.pt"))
        print(f"  ✓ best saved (val MRR={best_mrr:.4f})")




✓ Training on cuda for 20 epochs
  loss = InfoNCE(τ=0.05) + 0.3 * MSE(cos, jaccard)


Epoch 01/20: 100%|██████████| 118/118 [00:45<00:00,  2.58it/s, ce=2.2902, loss=2.2919, reg=0.0058]


Epoch 01 | train_loss=2.9242 (ce=2.9216 reg=0.0086) | val MRR=0.2636  Hits@1=0.1890  Hits@5=0.3291
  ✓ best saved (val MRR=0.2636)


Epoch 02/20: 100%|██████████| 118/118 [00:45<00:00,  2.60it/s, ce=1.3414, loss=1.3431, reg=0.0055]


Epoch 02 | train_loss=2.0807 (ce=2.0772 reg=0.0118) | val MRR=0.2839  Hits@1=0.1996  Hits@5=0.3567
  ✓ best saved (val MRR=0.2839)


...

Epoch 18 | train_loss=0.8773 (ce=0.8712 reg=0.0202) | val MRR=0.4222  Hits@1=0.3163  Hits@5=0.5244
  ✓ best saved (val MRR=0.4222)


Epoch 19/20: 100%|██████████| 118/118 [00:45<00:00,  2.60it/s, ce=0.9774, loss=0.9819, reg=0.0149]


Epoch 19 | train_loss=0.8307 (ce=0.8244 reg=0.0211) | val MRR=0.4102  Hits@1=0.3015  Hits@5=0.5053


Epoch 20/20: 100%|██████████| 118/118 [00:45<00:00,  2.59it/s, ce=0.6554, loss=0.6611, reg=0.0192]


Epoch 20 | train_loss=0.7943 (ce=0.7876 reg=0.0224) | val MRR=0.4244  Hits@1=0.3248  Hits@5=0.5011
  ✓ best saved (val MRR=0.4244)


In [12]:

# ============================================================================
# 8. Final evaluation on the test split
# ============================================================================
print(f"\n✓ Training complete. Best val MRR: {best_mrr:.4f}")
print("✓ Reloading best checkpoint and evaluating on test split...")

best = torch.load(os.path.join(CKPT_DIR, "best_model.pt"), map_location=device)
model.load_state_dict(best["model_state_dict"])
test = evaluate_ranking(model, test_loader, device)
print(f"TEST  MRR={test['mrr']:.4f}  Hits@1={test['hits@1']:.4f}  Hits@5={test['hits@5']:.4f}")


✓ Training complete. Best val MRR: 0.4244
✓ Reloading best checkpoint and evaluating on test split...
TEST  MRR=0.4487  Hits@1=0.3411  Hits@5=0.5572


In [13]:
# ============================================================================
# 9. Save the Final Model
# ============================================================================
# Create a 'models' directory in your project root if it doesn't exist
save_dir = os.path.join(PROJECT_ROOT, "sparqlce/models")
os.makedirs(save_dir, exist_ok=True)

# Define the final path (you can add a timestamp or metric to the filename)
final_model_path = os.path.join(save_dir, f"final_model_mrr_{test['mrr']:.4f}.pt")

# Save the weights
torch.save(model.state_dict(), final_model_path)
print(f"✓ Final model successfully saved to: {final_model_path}")

✓ Final model successfully saved to: /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/sparqlce/models/final_model_mrr_0.4487.pt
